# Helmholtz Equation Example

This notebook demonstrates how to solve the Helmholtz equation using `torch-fem`.

In [ ]:
import torch
from torchfem.acoustics.helmholtz import Helmholtz
from torchfem.mesh import rect_quad

# Create a 2D mesh
nodes, elements = rect_quad(10, 10)
nodes = nodes.to(torch.complex128)

# Set up the Helmholtz problem
k = 1.0
problem = Helmholtz(nodes, elements, k)

# Define analytical solution (plane wave)
p_analytical = torch.exp(-1j * k * nodes[:, 0].real)

# Apply Dirichlet boundary conditions
problem.constraints[nodes[:, 0] == 0.0, 0] = True
problem.constraints[nodes[:, 0] == 1.0, 0] = True
problem.displacements[nodes[:, 0] == 0.0, 0] = p_analytical[nodes[:, 0] == 0.0]
problem.displacements[nodes[:, 0] == 1.0, 0] = p_analytical[nodes[:, 0] == 1.0]

# Solve the problem
p_fem = problem.solve()

# Plot the solution
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.tricontourf(nodes[:, 0].real.numpy(), nodes[:, 1].real.numpy(), p_fem.real.numpy().ravel(), cmap='viridis')
plt.colorbar(label='Re(p)')
plt.title('FEM Solution (Real Part)')
plt.xlabel('x')
plt.ylabel('y')
plt.subplot(1, 2, 2)
plt.tricontourf(nodes[:, 0].real.numpy(), nodes[:, 1].real.numpy(), p_analytical.real.numpy().ravel(), cmap='viridis')
plt.colorbar(label='Re(p)')
plt.title('Analytical Solution (Real Part)')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()